# BAB 4 — Skenario 3: Studi Ablasi (Ablation Study)

Notebook ini menjalankan **tiga eksperimen ablasi** untuk mengukur kontribusi nyata setiap modul atensi dalam arsitektur AttentiveSkel-3D. Setiap ablasi menonaktifkan **satu modul** secara selektif, sementara modul lainnya tetap aktif.

| Skenario | Konfigurasi | File Tersimpan |
|---|---|---|
| **Baseline** *(referensi)* | `use_attention=False` — 3D-CNN murni | `baseline_3dcnn_model.pth` |
| **Ablasi A** — Tanpa Prior | `use_spatial_prior=False` | `ablasi_a_no_prior.pth` |
| **Ablasi B** — Tanpa Learned Spatial | `use_learned_spatial=False` | `ablasi_b_no_learned.pth` |
| **Ablasi C** — Tanpa Temporal | `use_temporal_attention=False` | `ablasi_c_no_temporal.pth` |
| **Model Penuh** *(upper-bound)* | Semua modul aktif | `best_model.pth` |

### Deskripsi Setiap Ablasi

- **Ablasi A — Tanpa Biomechanical Spatial Prior (BSP)**  
  Modul `use_spatial_prior=False`: parameter bobot per-sendi tidak diterapkan sebelum conv block. Jika akurasi turun signifikan, artinya inductive bias biomekanikal pada level raw skeleton memang berkontribusi penting.

- **Ablasi B — Tanpa Learned Spatial Attention**  
  Modul `use_learned_spatial=False`: Channel Attention (SE-style) setelah `conv_block_3` dimatikan. Mengukur kontribusi pemilihan channel adaptif setelah ekstraksi fitur 3D.

- **Ablasi C — Tanpa Temporal Attention**  
  Modul `use_temporal_attention=False`: Pembobotan frame berbasis softmax dimatikan. Mengukur apakah model benar-benar memanfaatkan frame-frame kritis dalam sekuens gerak.

> **Parameter umum semua ablasi:** Adam `lr=1e-3`, CrossEntropyLoss, 100 epoch, batch_size=16, split 70/15/15.

In [1]:
# ============================================================
# Cell 2: Import library, konfigurasi device, dan muat dataset
# ============================================================
import sys
from pathlib import Path

# Tambahkan root proyek ke sys.path agar modul src/ dapat diimport
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn

# Modul proyek
from src.data.dataset import create_dataloaders
from src.models.model_3dcnn import AttentiveSkel3D, count_parameters
from src.models.train import train_model

# Konfigurasi device: gunakan GPU jika tersedia, fallback ke CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[INFO] PyTorch  : {torch.__version__}")
print(f"[INFO] Device   : {device}")
if device.type == "cuda":
    print(f"[INFO] GPU      : {torch.cuda.get_device_name(0)}")

# ── Muat Dataset ──────────────────────────────────────────
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_manifest.csv"
SAVE_DIR      = PROJECT_ROOT / "models" / "saved_models"

train_loader, val_loader, test_loader = create_dataloaders(
    csv_file    = MANIFEST_PATH,
    batch_size  = 16,
    train_ratio = 0.70,
    val_ratio   = 0.15,
    num_workers = 0,       # 0 di Windows untuk menghindari masalah multiprocessing
    random_seed = 42,
)

n_train = len(train_loader.dataset)
n_val   = len(val_loader.dataset)
n_test  = len(test_loader.dataset)
print(f"\n[INFO] Total sampel       : {n_train + n_val + n_test}")
print(f"[INFO] Train / Val / Test : {n_train} / {n_val} / {n_test}")
print(f"[INFO] Iterasi per epoch  : {len(train_loader)}")

# ── Loss Function (dipakai bersama oleh semua ablasi) ─────
criterion = nn.CrossEntropyLoss()
print(f"\n[INFO] Criterion          : CrossEntropyLoss")
print(f"[INFO] Save dir           : {SAVE_DIR}")

[INFO] PyTorch  : 2.5.1
[INFO] Device   : cuda
[INFO] GPU      : NVIDIA GeForce RTX 3060 Ti
Dataset split selesai (seed=42):
  Train  :  340 sampel → 21 batch
  Val    :   73 sampel → 5 batch
  Test   :   74 sampel → 5 batch

[INFO] Total sampel       : 487
[INFO] Train / Val / Test : 340 / 73 / 74
[INFO] Iterasi per epoch  : 21

[INFO] Criterion          : CrossEntropyLoss
[INFO] Save dir           : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\models\saved_models


In [2]:
# ============================================================
# Cell 3: ABLASI A — Tanpa Biomechanical Spatial Prior (BSP)
# ============================================================
# use_spatial_prior=False : BSP tidak diterapkan ke tensor sebelum conv blocks.
# Modul lain (Learned Spatial + Temporal) tetap AKTIF.
# ──────────────────────────────────────────────────────────
print("=" * 60)
print("  ABLASI A — Tanpa Biomechanical Spatial Prior (BSP)")
print("=" * 60)

model_a = AttentiveSkel3D(
    num_classes          = 2,
    use_spatial_prior    = False,   # ← BSP dimatikan
    use_learned_spatial  = True,
    use_temporal_attention = True,
).to(device)

print(f"[INFO] use_spatial_prior      : {model_a.use_spatial_prior}")
print(f"[INFO] use_learned_spatial    : {model_a.use_learned_spatial}")
print(f"[INFO] use_temporal_attention : {model_a.use_temporal_attention}")
print(f"[INFO] Parameter trainable    : {count_parameters(model_a):,}")

optimizer_a = torch.optim.Adam(model_a.parameters(), lr=1e-3)

history_a = train_model(
    model         = model_a,
    train_loader  = train_loader,
    val_loader    = val_loader,
    criterion     = criterion,
    optimizer     = optimizer_a,
    num_epochs    = 100,
    device        = device,
    save_dir      = SAVE_DIR,
    save_filename = "ablasi_a_no_prior.pth",
    verbose       = True,
)

print(f"\n[DONE — ABLASI A]")
print(f"  Epoch terbaik   : {history_a['best_epoch']}")
print(f"  Val Loss terbaik: {history_a['best_val_loss']:.4f}")
print(f"  Val Acc terbaik : {history_a['val_acc'][history_a['best_epoch']-1]*100:.2f}%")
print(f"  Model disimpan  : {SAVE_DIR / 'ablasi_a_no_prior.pth'}")

  ABLASI A — Tanpa Biomechanical Spatial Prior (BSP)
[INFO] use_spatial_prior      : False
[INFO] use_learned_spatial    : True
[INFO] use_temporal_attention : True
[INFO] Parameter trainable    : 110,339
  Memulai pelatihan AttentiveSkel-3D
  Device    : cuda
  Epochs    : 100
  Save path : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\models\saved_models\ablasi_a_no_prior.pth
Epoch [  1/100] ✓ | Train Loss: 0.6976 | Train Acc:  43.45% | Val Loss: 0.6827 | Val Acc:  58.90% | Waktu: 0.6s
Epoch [  2/100] ✓ | Train Loss: 0.6664 | Train Acc:  72.32% | Val Loss: 0.6066 | Val Acc:  76.71% | Waktu: 0.3s
Epoch [  3/100] ✓ | Train Loss: 0.5871 | Train Acc:  83.04% | Val Loss: 0.5448 | Val Acc:  82.19% | Waktu: 0.2s
Epoch [  4/100] ✓ | Train Loss: 0.4745 | Train Acc:  86.90% | Val Loss: 0.4433 | Val Acc:  83.56% | Waktu: 0.2s
Epoch [  5/100] ✓ | Train Loss: 0.3573 | Train Acc:  89.88% | Val Loss: 0.3717 | Val Acc:  86.30% | Waktu: 0.2s
Epoch [  6/100]   | Train Loss: 0.3144 

In [3]:
# ============================================================
# Cell 4: ABLASI B — Tanpa Learned Spatial Attention
# ============================================================
# use_learned_spatial=False : Channel Attention (SE-style) setelah conv_block_3
#                             dinonaktifkan.
# Modul lain (BSP + Temporal) tetap AKTIF.
# ──────────────────────────────────────────────────────────
print("=" * 60)
print("  ABLASI B — Tanpa Learned Spatial Attention")
print("=" * 60)

model_b = AttentiveSkel3D(
    num_classes          = 2,
    use_spatial_prior    = True,
    use_learned_spatial  = False,   # ← Learned Spatial dimatikan
    use_temporal_attention = True,
).to(device)

print(f"[INFO] use_spatial_prior      : {model_b.use_spatial_prior}")
print(f"[INFO] use_learned_spatial    : {model_b.use_learned_spatial}")
print(f"[INFO] use_temporal_attention : {model_b.use_temporal_attention}")
print(f"[INFO] Parameter trainable    : {count_parameters(model_b):,}")

optimizer_b = torch.optim.Adam(model_b.parameters(), lr=1e-3)

history_b = train_model(
    model         = model_b,
    train_loader  = train_loader,
    val_loader    = val_loader,
    criterion     = criterion,
    optimizer     = optimizer_b,
    num_epochs    = 100,
    device        = device,
    save_dir      = SAVE_DIR,
    save_filename = "ablasi_b_no_learned.pth",
    verbose       = True,
)

print(f"\n[DONE — ABLASI B]")
print(f"  Epoch terbaik   : {history_b['best_epoch']}")
print(f"  Val Loss terbaik: {history_b['best_val_loss']:.4f}")
print(f"  Val Acc terbaik : {history_b['val_acc'][history_b['best_epoch']-1]*100:.2f}%")
print(f"  Model disimpan  : {SAVE_DIR / 'ablasi_b_no_learned.pth'}")

  ABLASI B — Tanpa Learned Spatial Attention
[INFO] use_spatial_prior      : True
[INFO] use_learned_spatial    : False
[INFO] use_temporal_attention : True
[INFO] Parameter trainable    : 102,020
  Memulai pelatihan AttentiveSkel-3D
  Device    : cuda
  Epochs    : 100
  Save path : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\models\saved_models\ablasi_b_no_learned.pth
Epoch [  1/100] ✓ | Train Loss: 0.6635 | Train Acc:  66.07% | Val Loss: 0.6515 | Val Acc:  60.27% | Waktu: 0.2s
Epoch [  2/100] ✓ | Train Loss: 0.6046 | Train Acc:  72.62% | Val Loss: 0.5747 | Val Acc:  76.71% | Waktu: 0.2s
Epoch [  3/100] ✓ | Train Loss: 0.4948 | Train Acc:  81.85% | Val Loss: 0.4731 | Val Acc:  82.19% | Waktu: 0.2s
Epoch [  4/100] ✓ | Train Loss: 0.4009 | Train Acc:  88.39% | Val Loss: 0.4400 | Val Acc:  87.67% | Waktu: 0.2s
Epoch [  5/100]   | Train Loss: 0.3387 | Train Acc:  90.18% | Val Loss: 0.4637 | Val Acc:  82.19% | Waktu: 0.2s
Epoch [  6/100] ✓ | Train Loss: 0.2922 | Trai

In [4]:
# ============================================================
# Cell 5: ABLASI C — Tanpa Temporal Attention
# ============================================================
# use_temporal_attention=False : Pembobotan frame berbasis softmax dinonaktifkan.
# Modul lain (BSP + Learned Spatial) tetap AKTIF.
# ──────────────────────────────────────────────────────────
print("=" * 60)
print("  ABLASI C — Tanpa Temporal Attention")
print("=" * 60)

model_c = AttentiveSkel3D(
    num_classes          = 2,
    use_spatial_prior    = True,
    use_learned_spatial  = True,
    use_temporal_attention = False,   # ← Temporal dimatikan
).to(device)

print(f"[INFO] use_spatial_prior      : {model_c.use_spatial_prior}")
print(f"[INFO] use_learned_spatial    : {model_c.use_learned_spatial}")
print(f"[INFO] use_temporal_attention : {model_c.use_temporal_attention}")
print(f"[INFO] Parameter trainable    : {count_parameters(model_c):,}")

optimizer_c = torch.optim.Adam(model_c.parameters(), lr=1e-3)

history_c = train_model(
    model         = model_c,
    train_loader  = train_loader,
    val_loader    = val_loader,
    criterion     = criterion,
    optimizer     = optimizer_c,
    num_epochs    = 100,
    device        = device,
    save_dir      = SAVE_DIR,
    save_filename = "ablasi_c_no_temporal.pth",
    verbose       = True,
)

print(f"\n[DONE — ABLASI C]")
print(f"  Epoch terbaik   : {history_c['best_epoch']}")
print(f"  Val Loss terbaik: {history_c['best_val_loss']:.4f}")
print(f"  Val Acc terbaik : {history_c['val_acc'][history_c['best_epoch']-1]*100:.2f}%")
print(f"  Model disimpan  : {SAVE_DIR / 'ablasi_c_no_temporal.pth'}")

  ABLASI C — Tanpa Temporal Attention
[INFO] use_spatial_prior      : True
[INFO] use_learned_spatial    : True
[INFO] use_temporal_attention : False
[INFO] Parameter trainable    : 110,243
  Memulai pelatihan AttentiveSkel-3D
  Device    : cuda
  Epochs    : 100
  Save path : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\models\saved_models\ablasi_c_no_temporal.pth
Epoch [  1/100] ✓ | Train Loss: 0.5588 | Train Acc:  76.19% | Val Loss: 0.4815 | Val Acc:  82.19% | Waktu: 0.2s
Epoch [  2/100] ✓ | Train Loss: 0.3838 | Train Acc:  86.31% | Val Loss: 0.4108 | Val Acc:  82.19% | Waktu: 0.2s
Epoch [  3/100] ✓ | Train Loss: 0.3015 | Train Acc:  88.99% | Val Loss: 0.3992 | Val Acc:  84.93% | Waktu: 0.2s
Epoch [  4/100]   | Train Loss: 0.2987 | Train Acc:  88.99% | Val Loss: 0.4527 | Val Acc:  83.56% | Waktu: 0.2s
Epoch [  5/100]   | Train Loss: 0.3062 | Train Acc:  87.80% | Val Loss: 0.4068 | Val Acc:  78.08% | Waktu: 0.2s
Epoch [  6/100]   | Train Loss: 0.2966 | Train Acc:

In [5]:
# ============================================================
# Cell 6: Komparasi Hasil Ketiga Ablasi
# ============================================================
# Tampilkan ringkasan Best Val Accuracy & Best Val Loss dari
# history_a, history_b, dan history_c secara side-by-side.
# ──────────────────────────────────────────────────────────

def _best_acc(history):
    return history["val_acc"][history["best_epoch"] - 1] * 100

results = [
    {
        "Skenario"         : "Ablasi A — Tanpa BSP",
        "Dimatikan"        : "use_spatial_prior=False",
        "Best Epoch"       : history_a["best_epoch"],
        "Best Val Loss"    : history_a["best_val_loss"],
        "Best Val Acc (%)" : _best_acc(history_a),
    },
    {
        "Skenario"         : "Ablasi B — Tanpa Learned Spatial",
        "Dimatikan"        : "use_learned_spatial=False",
        "Best Epoch"       : history_b["best_epoch"],
        "Best Val Loss"    : history_b["best_val_loss"],
        "Best Val Acc (%)" : _best_acc(history_b),
    },
    {
        "Skenario"         : "Ablasi C — Tanpa Temporal",
        "Dimatikan"        : "use_temporal_attention=False",
        "Best Epoch"       : history_c["best_epoch"],
        "Best Val Loss"    : history_c["best_val_loss"],
        "Best Val Acc (%)" : _best_acc(history_c),
    },
]

# ── Cetak tabel komparasi ─────────────────────────────────
col_w = [38, 30, 12, 16, 18]
header = ["Skenario", "Dimatikan", "Best Epoch", "Best Val Loss", "Best Val Acc (%)"]
sep    = "+" + "+".join("-" * w for w in col_w) + "+"
row_fmt = "|" + "|".join(f"{{:<{w}}}" for w in col_w) + "|"

print(f"\n{'='*120}")
print(f"  KOMPARASI HASIL ABLATION STUDY")
print(f"{'='*120}")
print(sep)
print(row_fmt.format(*header))
print(sep)
for r in results:
    print(row_fmt.format(
        r["Skenario"],
        r["Dimatikan"],
        str(r["Best Epoch"]),
        f"{r['Best Val Loss']:.4f}",
        f"{r['Best Val Acc (%)']:.2f}%",
    ))
print(sep)

print("\nInterpretasi:")
best_ablasi = max(results, key=lambda r: r["Best Val Acc (%)"])
worst_ablasi = min(results, key=lambda r: r["Best Val Acc (%)"])
print(f"  Ablasi dengan akurasi TERTINGGI : {best_ablasi['Skenario']}")
print(f"  Ablasi dengan akurasi TERENDAH  : {worst_ablasi['Skenario']}")
print(f"  → Modul dengan dampak paling besar jika dimatikan: {worst_ablasi['Dimatikan']}")


  KOMPARASI HASIL ABLATION STUDY
+--------------------------------------+------------------------------+------------+----------------+------------------+
|Skenario                              |Dimatikan                     |Best Epoch  |Best Val Loss   |Best Val Acc (%)  |
+--------------------------------------+------------------------------+------------+----------------+------------------+
|Ablasi A — Tanpa BSP                  |use_spatial_prior=False       |31          |0.2183          |95.89%            |
|Ablasi B — Tanpa Learned Spatial      |use_learned_spatial=False     |34          |0.1581          |98.63%            |
|Ablasi C — Tanpa Temporal             |use_temporal_attention=False  |53          |0.1343          |95.89%            |
+--------------------------------------+------------------------------+------------+----------------+------------------+

Interpretasi:
  Ablasi dengan akurasi TERTINGGI : Ablasi B — Tanpa Learned Spatial
  Ablasi dengan akurasi TERENDAH  :